# Notebook 2 – Data Inspection

**Sprint 4 – Data Inspection & Exploratory Data Analysis**  
Workflow: `Load → Inspect → Understand → Analyze → Visualize → Identify Problems → Document Findings`  
**Dataset:** Telco Customer Churn (IBM sample data, 7,043 customers × 21 columns)

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Load the RAW dataset (nothing has been modified yet).
# This loader checks common local locations first, then tries the public source.
from pathlib import Path

TELCO_URLS = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/main/data/Telco-Customer-Churn.csv",
]

candidate_names = [
    "telco.csv",
    "Telco-Customer-Churn.csv",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
]

search_roots = [Path.cwd(), *list(Path.cwd().parents)[:4]]
local_candidates = []
for root in search_roots:
    for name in candidate_names:
        local_candidates.extend([
            root / name,
            root / "data" / name,
            root / "Datasets" / name,
            root / "dataset" / name,
        ])

local_path = next((p.resolve() for p in local_candidates if p.is_file()), None)

if local_path is not None:
    df_raw = pd.read_csv(local_path)
    print("Loaded local dataset:", local_path)
else:
    load_errors = []
    df_raw = None
    for url in TELCO_URLS:
        try:
            df_raw = pd.read_csv(url)
            print("Loaded dataset from public source.")
            break
        except Exception as exc:
            load_errors.append(f"{url} -> {type(exc).__name__}: {exc}")
    if df_raw is None:
        raise FileNotFoundError(
            "Could not find the Telco dataset locally, and downloading it failed. "
            "Place the original Telco Customer Churn CSV in a 'data' folder next to "
            "your notebooks (or in the parent folder's 'data' folder), name it "
            "'telco.csv' or 'Telco-Customer-Churn.csv', and rerun this cell. "
            "If using a URL, check your internet connection.\n\nDetails:\n"
            + "\n".join(load_errors)
        )

df = df_raw.copy()
print("Loaded:", df.shape)


Loaded: (7043, 21)


### Topic Explanation

This topic checks dataset size, field names, and row indexing to understand its structure.

## 1. Number of Rows, Columns, Shape, Column Names and Index

### 1. Concept
`shape` returns (rows, columns). `columns` lists the column labels. `index` is the row label axis (a `RangeIndex` by default).

### 2. Example
*Simple:* counting the rows and column headers of a spreadsheet.  
*Business:* confirm that the extract contains all expected customers and fields.  
*AI/ML use:* first sanity check that the data matches the data contract.

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [2]:
print("Shape         :", df.shape)
print("Rows          :", df.shape[0])
print("Columns       :", df.shape[1])
print("\nColumn names  :", list(df.columns))
print("\nIndex         :", df.index)

Shape         : (7043, 21)
Rows          : 7043
Columns       : 21

Column names  : ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Index         : RangeIndex(start=0, stop=7043, step=1)


### 5. Interpretation
The frame has 7,043 rows × 21 columns and a default `RangeIndex(0..7042)`. Column names are clean (no spaces) but use mixed styles (`customerID`, `tenure`, `PaperlessBilling`).

### 6. Insight
The index carries no business meaning – the real key is `customerID`.

### 7. AI/ML Relevance
Consistent naming and a known key avoid join/merge errors later in the pipeline.

### Topic Explanation

This topic previews the first, last, and randomly selected records to spot structure and unusual values.

## 2. Looking at the Data: head(), tail(), sample()

### 1. Concept
`head()` shows the first rows, `tail()` the last rows, `sample()` random rows. Random samples avoid the bias of sorted files.

### 2. Example
*Simple:* skim the first and last pages of a book plus a random page.  
*Business:* spot obviously wrong values or formatting.  
*AI/ML use:* detect sorting/ordering that could cause leakage in train-test splits.

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [3]:
display(df.head())
display(df.tail())
display(df.sample(5, random_state=42))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


### 5. Interpretation
The rows look like valid customer records. Values are text labels (`Yes`/`No`, plan names) and numbers (charges, tenure).

### 6. Insight
Head, tail and a random sample look alike, so the file does not appear to be sorted by any column.

### 7. AI/ML Relevance
If data were sorted by the target, a non-shuffled split would train and test on different distributions.

### Topic Explanation

This topic checks each column's stored data type and non-null count so data can be interpreted correctly.

## 3. Data Types: dtypes and info()

### 1. Concept
`dtypes` shows the storage type of each column; `info()` also shows non-null counts and memory usage. Types in pandas: `int64`, `float64`, `object/str`, `bool`, `datetime64`, `category`.

### 2. Example
*Simple:* a number written as text cannot be added.  
*Business:* a bill amount stored as text cannot be summed for revenue.  
*AI/ML use:* ML libraries need numeric input; wrong dtypes break or distort preprocessing.

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [4]:
df.info()
print()
print(df.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

### 5. Interpretation
Most columns are text (`object`/`str`), 2 are `int64` (`SeniorCitizen`, `tenure`) and 1 is `float64` (`MonthlyCharges`). **`TotalCharges` is text although it contains money amounts.** `info()` shows no missing values, which is suspicious given this type mismatch.

### 6. Insight
Because `TotalCharges` is text, `info()` cannot see its hidden blank values (they are empty strings, not NaN). This is investigated in Notebook 3.

### 7. AI/ML Relevance
Converting types correctly is one of the first preprocessing tasks; `info()` is the fastest way to discover the need.

### Topic Explanation

This topic summarizes numeric columns with statistics such as count, mean, spread, minimum, and maximum.

## 4. Statistical Summary: describe()

### 1. Concept
`describe()` returns count, mean, std, min, quartiles and max for numerical columns; with `include='object'` it returns count, unique, top and frequency for text columns.

### 2. Example
*Simple:* a quick report card of a column.  
*Business:* typical monthly bill, typical customer age on the network.  
*AI/ML use:* reveals scales, ranges and suspicious values (negatives, huge maxima).

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [5]:
display(df.describe().T)
display(df.describe(include="object").T)

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


,count,unique,top,freq
customerID,7043,7043,7590-VHVEG,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


### 5. Interpretation
`tenure` ranges 0–72 months (mean ≈ 32, median 29); `MonthlyCharges` ranges about \$18–\$119 (mean ≈ \$65). `SeniorCitizen` mean ≈ 0.16 means ~16% are seniors (it is a flag). `describe` skips `TotalCharges` completely because it is text.

### 6. Insight
Features are on very different scales (tenure 0–72 vs charges up to ~119 and totals in the thousands). Mean ≈ median for tenure and charges suggests no extreme skew in those two.

### 7. AI/ML Relevance
Different scales matter for distance-based and gradient-based models (kNN, SVM, neural networks) → scaling is needed later.

### Topic Explanation

This topic counts distinct values and category frequencies to understand the values present in each column.

## 5. nunique() and value_counts()

### 1. Concept
`nunique()` counts distinct values per column; `value_counts()` shows the frequency of each value. Together they separate identifiers, categorical and continuous columns.

### 2. Example
*Simple:* how many different colours are in the box and how many of each.  
*Business:* how many customers are on each contract.  
*AI/ML use:* cardinality decides the encoding method (one-hot vs target/ordinal encoding).

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [6]:
print(df.nunique().sort_values(ascending=False))
print("\n---- value_counts for selected columns ----")
for c in ["Churn", "Contract", "InternetService", "PaymentMethod", "SeniorCitizen"]:
    print(f"\n{c}")
    print(df[c].value_counts())

customerID          7043
TotalCharges        6531
MonthlyCharges      1585
tenure                73
PaymentMethod          4
StreamingMovies        3
TechSupport            3
OnlineBackup           3
StreamingTV            3
DeviceProtection       3
MultipleLines          3
InternetService        3
OnlineSecurity         3
Contract               3
Partner                2
SeniorCitizen          2
gender                 2
Dependents             2
PhoneService           2
PaperlessBilling       2
Churn                  2
dtype: int64

---- value_counts for selected columns ----

Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Contract
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

PaymentMethod
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic

### 5. Interpretation
`customerID` has 7,043 unique values (identifier). `TotalCharges` has ~6,531 and `MonthlyCharges` ~1,585 unique values (continuous). `tenure` has 73. All remaining columns have 2–4 unique values (categorical).

### 6. Insight
Month-to-month is the biggest contract group; Electronic check is the most common payment method; Fiber optic is the most common internet service.

### 7. AI/ML Relevance
Low-cardinality categoricals are safe for one-hot encoding. The identifier must be dropped.

### Topic Explanation

This topic focuses on identifying column roles. Read the existing notes, follow the example cells, and explain what the resulting output tells you.

## 6. Identifying Column Roles

### 1. Concept
Classify each column as **numerical, categorical, date/time, identifier or potential target** using evidence (dtype, cardinality, content), not guesses.

### 2. Example
*Simple:* sorting kitchen items into shelves.  
*Business:* tells analysts which columns can be summed, grouped or ignored.  
*AI/ML use:* drives the preprocessing plan per column type.

### 3. Implementation & 4. Output
*(code below, output printed under each cell)*

In [7]:
def column_role(s, name):
    if name == "Churn":
        return "target (categorical, binary)"
    if s.nunique() == len(s):
        return "identifier"
    if name == "TotalCharges":
        return "numerical (stored as text!)"
    if name == "SeniorCitizen":
        return "categorical (binary, stored as int)"
    if pd.api.types.is_numeric_dtype(s):
        return "numerical (discrete)" if (s.nunique() < 100 and name == "tenure") else "numerical (continuous)"
    return "categorical"

roles = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "role": [column_role(df[c], c) for c in df.columns],
})
display(roles)

# date/time columns? -> try to detect
date_like = [c for c in df.columns if any(k in c.lower() for k in ("date", "time", "day", "month", "year"))]
print("\nColumns with date/time names:", date_like)
print("Datetime dtypes present:", df.select_dtypes(include="datetime").shape[1])
print("\nRole counts:")
print(roles["role"].value_counts())

,column,dtype,n_unique,role
0,customerID,str,7043,identifier
1,gender,str,2,categorical
2,SeniorCitizen,int64,2,"categorical (binary, stored as int)"
3,Partner,str,2,categorical
4,Dependents,str,2,categorical
5,tenure,int64,73,numerical (discrete)
6,PhoneService,str,2,categorical
7,MultipleLines,str,3,categorical
8,InternetService,str,3,categorical
9,OnlineSecurity,str,3,categorical



Columns with date/time names: ['MonthlyCharges']
Datetime dtypes present: 0

Role counts:
role
categorical                            15
identifier                              1
categorical (binary, stored as int)     1
numerical (discrete)                    1
numerical (continuous)                  1
numerical (stored as text!)             1
target (categorical, binary)            1
Name: count, dtype: int64


### 5. Interpretation
Role summary: **numerical** – `tenure`, `MonthlyCharges`, `TotalCharges`; **categorical** – 16 columns plus the target; **identifier** – `customerID`; **date/time** – none (`tenure` is a duration count, not a date); **potential target** – `Churn` (the only outcome-like column; could also model `MonthlyCharges` as a regression target for a different business question).

### 6. Insight
There is no time dimension, so time-series analysis and date feature engineering are not applicable; the dataset is a single snapshot.

### 7. AI/ML Relevance
A documented role table becomes the blueprint for the preprocessing sprint: drop the ID, encode categoricals, fix/scale numericals, treat the target separately.

## Summary of Notebook 2

| Check | Result |
|---|---|
| Shape | 7,043 × 21 |
| Missing values reported by `info()` | None (but `TotalCharges` is text and hides blanks) |
| Numerical columns | `tenure`, `MonthlyCharges`, `TotalCharges` |
| Categorical columns | 16 + target `Churn` (incl. `SeniorCitizen` stored as int) |
| Identifier | `customerID` |
| Date/time columns | None |
| Potential target | `Churn` |

**Issues to carry into Notebook 3:** `TotalCharges` dtype; hidden blanks; redundant category levels.